<a href="https://colab.research.google.com/github/Kirrrk-git/rise-unet-rzsm/blob/mindanao-adaptation/notebooks/06_mindanao_datacube_and_anomaly_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# RISE-UNet Step 21D.4: 11-Year Production RZSM Data Cube Compilation & Anomaly Pipeline

**Authoritative Parent Study**: Lesinger & Tian (2025), *Nature Communications*, DOI: `10.1038/s41467-025-62761-3`  
**Target Baseline Model**: **Mindanao Model A0** (Adapted from EX29 Recursive Hybrid RISE-UNet)  
**Git Branch**: `mindanao-adaptation`  
**Milestone**: Sub-Phase 21D (Step 21D.4-PREFLIGHT & Step 21D.4 Production Cube Compilation)  

### Production Pipeline Objectives:
1. **Step 21D.4-PREFLIGHT (Preflight Verification Gate)**: Audit all 265 NetCDF files covering the 12 Dec 2014–31 Dec 2025 support window ($96,912$ hourly timestamps across $4,038$ calendar days), verifying complete retention of layers `swvl1`, `swvl2`, `swvl3`, leap Februaries, and strict alignment with the frozen spatial contract.
2. **Depth-Weighted Integration & Land-Aware Remapping**: Calculate the 0–100 cm volumetric index ($\text{RZSM}_{0-100} = 0.07\cdot\text{SM}_1 + 0.21\cdot\text{SM}_2 + 0.72\cdot\text{SM}_3$) and remap to Candidate A $0.25^\circ$ grid ($32 \times 48$) via high-performance vectorized land-aware remapping with nearest-neighbor coastal fallback.
3. **Continuous Backward Trailing Rolling Mean**: Compute continuous 7-day backward trailing rolling average (`center=False`, zero future leakage) over the full $4,038$-day archive series, ensuring early 2015 cases have complete trailing rolling memory.
4. **Locked Model A0 Seasonal Climatology & Anomalies**: Fit 3-month seasonal climatology (`season`: DJF, MAM, JJA, SON) strictly on nominal training years $2015 \le \text{year} \le 2021$, and derive daily seasonal anomalies.
5. **Domain-Wide Active Scalar Normalization**: Fit min-max normalization bounds strictly over active evaluation cells ($M_{i,j}=1$) within the training period, standardizing anomalies to $[0, 1]$ while strictly zero-filling the 1,410 non-evaluation cells.
6. **Nominal Timeline Slicing**: Slice nominal production timeline: 01 Jan 2015 to 31 Dec 2025 ($4,018$ days; $506,268$ evaluation cell-days), preserving 2014 exclusively as antecedent support memory.
7. **Census Audit & Zero-NaN Certification**: Assert strictly Zero NaNs/Infs across all 506,268 nominal active evaluation points.
8. **CF-1.8 NetCDF Export & Cloud Lake Sync**: Export final production cube `era5_land_rzsm_production_2015_2025.nc` to `gs://rise-unet-rzsm/processed/rzsm/production/`.


### Step 0: Google Colab Setup & Runtime Initialization
Configures high-speed cloud runtime, detects Colab execution, clones active branch `mindanao-adaptation`, authenticates Google Cloud Storage access, and installs dependencies.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('--> Running in Google Colab environment.')
    from google.colab import auth
    auth.authenticate_user()
    
    # Clone repository or pull latest commits
    repo_path = Path('/content/rise-unet-rzsm')
    if not repo_path.exists():
        print('--> Cloning repository (branch: mindanao-adaptation)...')
        subprocess.run(['git', 'clone', '-b', 'mindanao-adaptation', '--depth', '1',
                        'https://github.com/Kirrrk-git/rise-unet-rzsm.git', str(repo_path)], check=True)
    else:
        print('--> Pulling latest repository commits from origin/mindanao-adaptation...')
        subprocess.run(['git', '-C', str(repo_path), 'pull', 'origin', 'mindanao-adaptation'], check=True)
        
    os.chdir(str(repo_path))
    sys.path.insert(0, str(repo_path))
    
    # Install required high-performance dependencies
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'xarray', 'netCDF4', 'scipy', 'gcsfs', 'matplotlib', 'pyyaml'], check=True)
else:
    print('--> Running in local workstation environment.')
    repo_path = Path.cwd()
    sys.path.insert(0, str(repo_path))

print(f'--> Working directory: {Path.cwd()}')


### Step 0b: Spatial Foundation Contract Verification
Assert the existence and integrity of the 5 frozen spatial foundation artifacts establishing Candidate A ($32 \times 48$) and the 126-cell binary evaluation mask ($f \ge 0.50$).


In [ ]:
import numpy as np
import xarray as xr
import yaml

contract_path = Path('contracts/spatial/spatial_grid_contract.yaml')
assert contract_path.exists(), f'Spatial contract missing at {contract_path}'

with open(contract_path, 'r') as f:
    spatial_contract = yaml.safe_load(f)

grid_nc = Path('processed/grid/mindanao_025deg.nc')
mask_nc = Path('processed/grid/mindanao_eval_mask_025.nc')
assert grid_nc.exists(), f'Grid NC missing at {grid_nc}'
assert mask_nc.exists(), f'Mask NC missing at {mask_nc}'

ds_grid = xr.open_dataset(grid_nc)
ds_mask = xr.open_dataset(mask_nc)

eval_mask = ds_mask['evaluation_mask'].values
n_eval = int(np.sum(eval_mask == 1))
n_buffer = int(np.sum(eval_mask == 0))

print(f'[PASS] Spatial grid verified: {ds_grid.dims["lat"]} lats x {ds_grid.dims["lon"]} lons (32 x 48)')
print(f'[PASS] Evaluation cells: {n_eval} active (expected 126), {n_buffer} zero-filled buffer (expected 1,410)')
assert n_eval == 126, f'Expected 126 evaluation cells, got {n_eval}'
assert n_buffer == 1410, f'Expected 1,410 buffer cells, got {n_buffer}'


### Step 1: Step 21D.4-PREFLIGHT (Production Preflight Verification Gate)
Executes the formal 11-point Preflight Gate before running large-scale tensor compilations:
- 1. Input File Census = Exactly 265 NetCDF files in `gs://rise-unet-rzsm/raw/era5_land/production/`
- 2. Temporal Coverage = 12 Dec 2014 to 31 Dec 2025 ($4,038$ continuous archive calendar days)
- 3. Variable Completeness = `swvl1`, `swvl2`, `swvl3` present across all archive files
- 4. Hourly Timestamp Continuity = 24 hourly records/day; exactly $96,912$ hourly timestamps across support period; leap days intact
- 5. Timestamp Hygiene = Monotonic time coordinate with zero duplicate timestamps
- 6. Grid Geometry Compatibility = Candidate A cell centers ($32 \times 48$, lat $11.75 \to 4.00$, lon $116.00 \to 127.75$)
- 7. Spatial Contract Consistency = Specifications, dimensions, and artifact hashes match frozen spatial artifacts
- 8. Training Period Isolation = Strictly locked to 2015–2021 (7 full calendar years)
- 9. Normalization Scope = Domain-wide scalar min-max bounds fitted strictly over active evaluation cells within 2015–2021
- 10. Antecedent Support Isolation = 2014 data (20 days) restricted to rolling memory initialization; never enters training statistics
- 11. Output CF-1.8 Specification = Variable names and global metadata attributes frozen


In [ ]:
from scripts.verify_step_21d4_preflight import run_full_preflight_gate

preflight_results = run_full_preflight_gate()
assert preflight_results['overall_preflight_passed'], 'Preflight Verification Gate FAILED! Halt compilation.'
print('\n[CERTIFIED PASS] Step 21D.4-PREFLIGHT verified! Cleared for production compilation.')


### Step 2: Full Production 11-Year RZSM Cube Compilation (Step 21D.4)
Executes the verified compilation engine `src/data/compile_cube.py` over the full 4,038 archive days:
1. Synchronizes the 265 NetCDF files from `gs://rise-unet-rzsm/raw/era5_land/production/`.
2. Computes 24-hour daily means and Kyle Lesinger depth weighting: $0.07\cdot\text{SM}_1 + 0.21\cdot\text{SM}_2 + 0.72\cdot\text{SM}_3$.
3. Vectorized land-aware remapping to Candidate A ($32 \times 48$) with nearest-neighbor coastal fallback.
4. Continuous 7-day backward trailing rolling mean (`center=False`, zero future leakage) over full archive series.
5. Locked Model A0 3-month seasonal climatology (DJF, MAM, JJA, SON) on training years $\le 2021$.
6. Seasonal anomalies and domain-wide active scalar min-max normalization on training fold active cells.
7. Slices nominal production period: 01 Jan 2015 to 31 Dec 2025 ($4,018$ days; $506,268$ evaluation cell-days).
8. Formal census audit asserting Zero NaNs/Infs over all 506,268 active evaluation points.


In [ ]:
import subprocess
from pathlib import Path
from src.data.compile_cube import (
    ProductionCubeConfig,
    compile_full_11yr_rzsm_cube,
)

config = ProductionCubeConfig()
print('--> Production Cube Compilation Configuration:')
print(f'    Nominal Training Period: {config.train_start_year} - {config.train_end_year}')
print(f'    Validation Period:       {config.val_years}')
print(f'    Held-Out Test Period:    {config.test_years}')
print(f'    Climatology Method:      {config.climatology_method} (Locked Model A0)')
print(f'    Expected Nominal Days:   {config.expected_nominal_days} days (506,268 evaluation samples)')
print(f'    Expected Archive Days:   {config.expected_archive_days} days (508,788 evaluation samples)')

# 1. Setup local archive storage directory
RAW_DIR = Path('/content/era5_land_raw') if IN_COLAB else Path('raw/era5_land/production')
RAW_DIR.mkdir(parents=True, exist_ok=True)

# 2. Synchronize 265 NetCDF files from GCS if not present locally
existing_nc = list(RAW_DIR.glob('*.nc'))
if len(existing_nc) < 265:
    print(f'--> Synchronizing 265 archive NetCDF files from GCS to {RAW_DIR}...')
    GCS_SOURCE = 'gs://rise-unet-rzsm/raw/era5_land/production/*.nc'
    if IN_COLAB:
        subprocess.run(f'gsutil -m cp {GCS_SOURCE} {RAW_DIR}/', shell=True, check=True)
    else:
        subprocess.run(f'gcloud storage cp {GCS_SOURCE} {RAW_DIR}/', shell=True, check=True)
    print(f'--> Successfully synchronized {len(list(RAW_DIR.glob("*.nc")))} NetCDF files.')
else:
    print(f'--> All 265 NetCDF files verified locally in {RAW_DIR}.')

# 3. Execute Production Cube Compilation Engine
OUTPUT_CUBE = Path('processed/rzsm/production/era5_land_rzsm_production_2015_2025.nc')
print('\n--> Launching full production data cube compilation pipeline...')
prod_ds, census = compile_full_11yr_rzsm_cube(
    archive_dir=RAW_DIR,
    output_path=OUTPUT_CUBE,
    config=config,
    slice_nominal_period=True,
    verbose=True,
)


### Step 3: Production Census Audit & Zero-NaN Certification
Verifies that the compiled production dataset strictly satisfies all completeness and numerical assertions:
- Zero NaNs/Infs over all 506,268 nominal active evaluation points
- 1,410 buffer/ocean cells strictly zero-filled
- Mandatory reporting standard satisfied: *“Zero NaNs/Infs across the 126 active evaluation cells; non-evaluation computational cells follow the frozen masking/zero-fill convention.”*


In [ ]:
print('================================================================================')
print('STEP 21D.4: FORMAL PRODUCTION CUBE CENSUS AUDIT RESULTS')
print('================================================================================')
for k, v in census.items():
    print(f'  {k}: {v}')

assert census['is_certified'], 'Production cube census audit FAILED!'
assert census['nans_active_cells'] == 0, f'Found {census["nans_active_cells"]} NaNs in active cells!'
assert census['infs_active_cells'] == 0, f'Found {census["infs_active_cells"]} Infs in active cells!'
assert census['nonzero_ocean_cells'] == 0, f'Found {census["nonzero_ocean_cells"]} nonzero ocean cells!'
assert census['total_active_evaluations'] == 506268, f'Expected 506,268 evaluation points, got {census["total_active_evaluations"]}'

print('\n' + '=' * 80)
print('[CERTIFIED PASS] Step 21D.4 Production Cube successfully compiled and certified!')
print(f'Standard: {census["mandatory_reporting_standard"]}')
print('=' * 80)


### Step 4: Publication Visual Verification Composite
Renders publication-grade 300 DPI 4-panel composite figure showing:
1. Multi-year Mean 0–100 cm RZSM spatial field across Mindanao active cells.
2. 11-Year continuous daily 7-day backward trailing rolling mean time series.
3. Locked Model A0 4-season climatology cycle (DJF, MAM, JJA, SON).
4. Standardized anomaly distribution $[0, 1]$ confirming domain-wide active scalar normalization.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Spatial Mean RZSM Field
mean_rzsm = prod_ds['rzsm_0_100_raw'].mean(dim='time').values
im0 = axes[0, 0].imshow(np.where(eval_mask == 1, mean_rzsm, np.nan), cmap='YlGnBu', origin='upper')
axes[0, 0].set_title('11-Year Mean 0-100 cm RZSM (Active Land Cells)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Longitude Grid Index (0-47)')
axes[0, 0].set_ylabel('Latitude Grid Index (0-31)')
plt.colorbar(im0, ax=axes[0, 0], fraction=0.035, pad=0.04, label='Volumetric SM (m3/m3)')

# Panel 2: Seasonal Climatology (DJF vs JJA)
clim_djf = prod_ds['climatology_seasonal'].sel(season='DJF').values
im1 = axes[0, 1].imshow(np.where(eval_mask == 1, clim_djf, np.nan), cmap='YlGnBu', origin='upper')
axes[0, 1].set_title('Locked Model A0 Climatological Mean: DJF', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Longitude Grid Index (0-47)')
axes[0, 1].set_ylabel('Latitude Grid Index (0-31)')
plt.colorbar(im1, ax=axes[0, 1], fraction=0.035, pad=0.04, label='Climatological Mean (m3/m3)')

# Panel 3: Domain-Averaged 11-Year Rolling Time Series
time_series = prod_ds['rzsm_0_100_rolling_7d'].values[:, eval_mask == 1].mean(axis=1)
axes[1, 0].plot(prod_ds['time'].values, time_series, color='navy', lw=0.8, alpha=0.85)
axes[1, 0].axvline(np.datetime64('2022-01-01'), color='crimson', linestyle='--', label='Validation Split (2022)')
axes[1, 0].axvline(np.datetime64('2024-01-01'), color='darkorange', linestyle='--', label='Test Split (2024)')
axes[1, 0].set_title('11-Year Daily 7-Day Trailing Rolling RZSM Series (2015-2025)', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Domain Mean RZSM (m3/m3)')
axes[1, 0].grid(True, linestyle='--', alpha=0.5)
axes[1, 0].legend(loc='upper right', frameon=True)

# Panel 4: Standardized Anomaly Distribution [0, 1]
norm_vals = prod_ds['rzsm_0_100_normalized'].values[:, eval_mask == 1].flatten()
axes[1, 1].hist(norm_vals, bins=50, color='teal', edgecolor='black', alpha=0.75, density=True)
axes[1, 1].set_title('Standardized Seasonal Anomaly Distribution [0, 1] (Active Cells)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Normalized Anomaly (dimensionless)')
axes[1, 1].set_ylabel('Probability Density')
axes[1, 1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
out_fig_dir = Path('figures')
out_fig_dir.mkdir(parents=True, exist_ok=True)
out_fig_path = out_fig_dir / 'mindanao_production_cube_verification_composite.png'
plt.savefig(out_fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'[PASS] Publication visual verification composite saved to {out_fig_path}')


### Step 5: NetCDF CF-1.8 Packaging & Cloud Lake Synchronization
Synchronizes final production data cube `era5_land_rzsm_production_2015_2025.nc` to `gs://rise-unet-rzsm/processed/rzsm/production/`.


In [ ]:
gcs_dest = 'gs://rise-unet-rzsm/processed/rzsm/production/era5_land_rzsm_production_2015_2025.nc'
print(f'--> Synchronizing production cube to Cloud Lake: {gcs_dest}...')
if IN_COLAB:
    subprocess.run(f'gsutil cp {OUTPUT_CUBE} {gcs_dest}', shell=True, check=True)
    subprocess.run(f'gsutil cp {out_fig_path} gs://rise-unet-rzsm/figures/', shell=True, check=False)
else:
    subprocess.run(f'gcloud storage cp {OUTPUT_CUBE} {gcs_dest}', shell=True, check=True)
    subprocess.run(f'gcloud storage cp {out_fig_path} gs://rise-unet-rzsm/figures/', shell=True, check=False)
print('\n[COMPLETE] 11-Year Production RZSM Data Cube and verification composite synchronized to Google Cloud Storage.')
